# RAGU as an LLM framework — building an LLM-as-a-judge

RAGU is usually introduced as a graph-RAG library, and that undersells it. Under
`KnowledgeGraph` sits a general-purpose LLM toolkit, and **none of it requires a
graph**:

| primitive | what it gives you |
|---|---|
| `LLM` / `LLMOpenAI` | text, Pydantic-validated structured output, streaming, and `batch_chat_completion` with concurrency + per-item error isolation |
| `Embedder` / `EmbedderOpenAI` | API-level batched embeddings, dimension auto-detection, automatic truncation to the model context window |
| `Scorer` / `ScorerOpenAI` / `ScorerCrossEncoder` | reranking behind one interface, remote or local |
| `ChatMessages` + `render()` | Jinja2 prompt templates that fan out over batch parameters in one call |
| `RAGUInstruction` | a prompt bound to its output schema, swappable at runtime |
| `CachedAsyncOpenAI` | rate limiting, retry with backoff, and an on-disk response cache — shared by every model above |

That is most of what an LLM application needs. This notebook builds a complete
evaluation harness out of those pieces.

**The harness:**

1. Grade answers against references with an LLM, using a Pydantic schema so the
   grades come back as validated numbers rather than prose.
2. Independently measure cosine similarity between answer and reference.
3. Combine the two into one score.

**Why both.** The LLM judge understands paraphrase and partial credit but is noisy
and can be talked into agreeing with a confident wrong answer. Cosine similarity is
cheap, stable and completely insensitive to whether a claim is *true* — it rewards
answers that merely look like the reference. Neither is trustworthy alone;
disagreement between them is the useful signal, so the report flags it instead of
hiding it inside the average.

**Environment:** `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`, and
optionally `OPENAI_BASE_URL`.

In [ ]:
import math
import os

from pydantic import BaseModel, Field

from ragu.common.global_parameters import Settings
from ragu.common.prompts import ChatMessages, SystemMessage, UserMessage
from ragu.common.prompts.messages import render
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI

# Weight of the LLM verdict in the combined score; the rest goes to cosine.
LLM_WEIGHT = 0.7

## 1. The dataset

A labelled set of questions, reference answers, and the answers under evaluation.
Where the predictions came from is irrelevant to the harness — another model, a
competitor's API, a human, or a RAG pipeline. Here they are literals, deliberately
containing one correct answer, one confidently wrong answer, and one that is
correct but incomplete.

In [ ]:
DATASET = [
    {
        "question": "Who created the C programming language?",
        "reference": "Dennis Ritchie created the C programming language at Bell Labs.",
        "prediction": "C was designed by Dennis Ritchie while he was working at Bell Labs.",
        "context": "Dennis Ritchie created C at Bell Labs in the early 1970s.",
    },
    {
        "question": "Who created the Python programming language?",
        "reference": "Guido van Rossum created Python.",
        # Confidently wrong: reads like the reference, states the wrong person.
        "prediction": "Python was created by Dennis Ritchie in the late 1980s.",
        "context": "Guido van Rossum began working on Python in December 1989.",
    },
    {
        "question": "What did Ken Thompson and Dennis Ritchie build together?",
        "reference": "They developed the Unix operating system at Bell Labs.",
        # Correct but incomplete: no mention of Unix being an operating system.
        "prediction": "They worked together at Bell Labs.",
        "context": "Ken Thompson and Dennis Ritchie developed Unix at Bell Labs.",
    },
]

print(f"{len(DATASET)} items to grade")

## 2. The verdict schema

`output_schema` on any `LLM` call turns free text into a validated object. Scores
are 1–5 integers rather than free-form floats: a small discrete scale is what LLMs
are actually reliable at, and it keeps runs comparable. Pydantic's `ge`/`le` are
enforced on parse, so an out-of-range score is a hard failure rather than a silent
outlier.

In [ ]:
class Judgement(BaseModel):
    """Structured verdict returned by the judge model."""

    correctness: int = Field(
        ge=1, le=5,
        description="Does the answer state the same facts as the reference? 5 = fully correct.",
    )
    completeness: int = Field(
        ge=1, le=5,
        description="Does the answer cover everything the reference covers? 5 = nothing missing.",
    )
    groundedness: int = Field(
        ge=1, le=5,
        description="Is every claim supported by the supplied context? 5 = fully supported.",
    )
    reasoning: str = Field(description="One or two sentences justifying the scores.")

## 3. The prompt

`ChatMessages` holds Jinja2 templates. Keeping the prompt as data rather than an
f-string is what makes it swappable, versionable, and renderable for inspection
before you spend a token on it.

In [ ]:
JUDGE_PROMPT = ChatMessages.from_messages([
    SystemMessage(
        "You are a strict evaluator of question answering. Grade only what is "
        "written. Do not reward fluent prose, and do not penalize an answer for "
        "wording that differs from the reference as long as the facts match. Judge "
        "groundedness against the supplied context alone: an answer that is correct "
        "but unsupported by the context still scores low on groundedness."
    ),
    UserMessage(
        "Question:\n{{ question }}\n\n"
        "Reference answer:\n{{ reference }}\n\n"
        "Answer under evaluation:\n{{ prediction }}\n\n"
        "Context the answer was generated from:\n{{ context }}"
    ),
])

# Inspect exactly what the model will see, before paying for it. The dataset rows
# already carry the four template variables, so they can be splatted straight in.
preview = render(JUDGE_PROMPT, **DATASET[0])[0]
print(preview.to_openai()[1]["content"])

## 4. The models

One `CachedAsyncOpenAI` is shared by the judge and the embedder, so rate limiting,
retries and caching are enforced once for the whole process.

`temperature=0` matters more for a judge than for a generator: an evaluation that
moves between runs cannot be compared between runs. The on-disk cache makes a
re-run of an unchanged evaluation free, which is what lets you iterate on the
*prompt* without re-paying for the whole set.

In [ ]:
Settings.language = "english"
Settings.cache_path = "ragu_working_dir/judge_cache"

client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
    retry_times_sec=(2, 4, 8),
)

judge = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"], temperature=0.0)
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

print(f"judge:    {judge.model_name}")
print(f"embedder: {embedder.model_name} (dim {embedder.dim})")

## 5. Grading

`render()` fans one template out over batch parameters — pass lists, get one
rendered conversation per row. `batch_chat_completion` then runs them concurrently
under the client's rate limit, with `continue_on_error=True` so one malformed
verdict returns `None` instead of losing the whole run.

In [ ]:
questions = [item["question"] for item in DATASET]
references = [item["reference"] for item in DATASET]
predictions = [item["prediction"] for item in DATASET]
contexts = [item["context"] for item in DATASET]

conversations = render(
    JUDGE_PROMPT,
    question=questions,
    reference=references,
    prediction=predictions,
    context=contexts,
)

verdicts = await judge.batch_chat_completion(
    [conversation.to_openai() for conversation in conversations],
    output_schema=Judgement,
    desc="Judging",
    continue_on_error=True,
)

print(f"{sum(v is not None for v in verdicts)}/{len(verdicts)} verdicts parsed")

## 6. The embedding half

Predictions and references go in one batch — `batch_embed_text` packs them into
API-level sub-batches, so this is one request rather than six.

In [ ]:
def cosine(left: list[float], right: list[float]) -> float:
    dot = sum(a * b for a, b in zip(left, right))
    norm = math.sqrt(sum(a * a for a in left)) * math.sqrt(sum(b * b for b in right))
    return dot / norm if norm else 0.0


embeddings = await embedder.batch_embed_text(predictions + references, desc="Embedding")
similarities = [
    cosine(prediction, reference)
    for prediction, reference in zip(embeddings[:len(predictions)], embeddings[len(predictions):])
]

for item, similarity in zip(DATASET, similarities):
    print(f"{similarity:.3f}  {item['question']}")

## 7. The report

Watch the second item: a wrong answer that mimics the reference's shape scores high
on cosine and low with the judge. That disagreement is the entire reason for
combining two metrics rather than trusting either.

In [ ]:
combined_scores = []
for item, verdict, similarity in zip(DATASET, verdicts, similarities):
    if verdict is None:
        print(f"\nQ: {item['question']}\n  judge FAILED — excluded")
        continue

    # Map the 1-5 scale onto 0-1 so it can be mixed with cosine similarity.
    llm_score = (verdict.correctness + verdict.completeness + verdict.groundedness - 3) / 12
    combined = LLM_WEIGHT * llm_score + (1 - LLM_WEIGHT) * similarity
    combined_scores.append(combined)

    print(f"\nQ: {item['question']}")
    print(f"  prediction: {item['prediction']}")
    print(f"  judge:      correctness={verdict.correctness} "
          f"completeness={verdict.completeness} groundedness={verdict.groundedness}"
          f" -> {llm_score:.3f}")
    print(f"  cosine:     {similarity:.3f}")
    print(f"  combined:   {combined:.3f}")
    print(f"  rationale:  {verdict.reasoning}")
    if abs(llm_score - similarity) > 0.3:
        print("  ** metrics disagree - inspect this one by hand **")

print(f"\nmean combined score: {sum(combined_scores) / len(combined_scores):.3f}")

## 8. Re-weight without re-grading

The verdicts and similarities are already in memory, so sweeping the weight costs
nothing. Use it to see how much your ranking depends on a number you picked by hand.

In [ ]:
graded = [(v, s) for v, s in zip(verdicts, similarities) if v is not None]
for weight in (0.0, 0.3, 0.5, 0.7, 1.0):
    scores = [
        weight * ((v.correctness + v.completeness + v.groundedness - 3) / 12) + (1 - weight) * s
        for v, s in graded
    ]
    print(f"llm_weight={weight:.1f} -> mean {sum(scores) / len(scores):.3f}")

## 9. Grading a RAG pipeline

Nothing above knew about retrieval. To evaluate a RAG system instead of literals,
swap where `prediction` and `context` come from — the rest of the harness is
unchanged:

```python
from ragu import KnowledgeGraph, LocalSearchEngine   # ...and the usual setup

responses  = await engine.batch_query(questions, LocalParams(top_k=10))
predictions = [str(r.response) for r in responses]
contexts    = [r.retrieval.to_text()[:4000] for r in responses]
```

`SearchEngineResponse` carries both the answer and the retrieval it came from,
which is why `groundedness` can separate a *generation* failure (the context had
the answer, the model ignored it) from a *retrieval* failure (the context never
contained it). That distinction is the main reason to feed the judge the context
at all.

## Other things the same primitives build

The pattern here — Pydantic schema, `ChatMessages` template, `render()` over a
batch, `batch_chat_completion` — is not specific to evaluation:

| task | how |
|---|---|
| Classification / labelling | schema with a `Literal[...]` field |
| Structured extraction from documents | schema of nested models; `ragu.parsers` for the input side |
| Batch summarization | `output_schema=str`, one row per document |
| Query rewriting or expansion | see `QueryPlanEngine` for a worked DAG version |
| Synthetic data generation | batch over seed parameters, validate on parse |
| Pairwise model comparison | two predictions per row, schema with a `winner` field |

All of them get the rate limiting, retries, caching and error isolation for free,
because those live in `CachedAsyncOpenAI` rather than in each task.